#### Name: Shashank Goel
#### Batch: B.Tech Data Science
#### Roll no: J011
#### Subject: NNDL

In [1]:
# Imports
import torch
import torch.nn as nn
import tensorflow as tf 
from tensorflow import keras 

In [2]:
# Check if CUDA is available
print(torch.cuda.is_available())
print(device := torch.device("cuda" if torch.cuda.is_available() else "cpu"))

True
cuda


In tensorflow, we make models using the following syntax:

model=keras.Sequential()
model.add(keras.layers.Dense(64, activation='relu', input_shape(32,)))

The syntax for using models in Pytorch remains similar.

In [3]:
model=nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 10),
    nn.Softmax(dim=1)
).to(device)

print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=128, bias=True)
  (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (6): ReLU()
  (7): Linear(in_features=64, out_features=10, bias=True)
  (8): Softmax(dim=1)
)


In [4]:
with torch.no_grad():
    scorer=model(torch.randn(10,1,28,28).to(device))

print(scorer.shape)
print(scorer)

torch.Size([10, 10])
tensor([[0.1208, 0.0481, 0.1140, 0.0956, 0.1214, 0.1396, 0.0686, 0.0554, 0.0901,
         0.1464],
        [0.0676, 0.0611, 0.1555, 0.0845, 0.0803, 0.1124, 0.0839, 0.0694, 0.1712,
         0.1141],
        [0.1117, 0.0321, 0.1378, 0.0869, 0.0770, 0.0750, 0.0865, 0.0595, 0.1498,
         0.1837],
        [0.0492, 0.0476, 0.1817, 0.1492, 0.1154, 0.0644, 0.0665, 0.0721, 0.1393,
         0.1147],
        [0.0696, 0.0411, 0.1016, 0.0863, 0.0808, 0.1434, 0.0713, 0.0472, 0.2327,
         0.1260],
        [0.1066, 0.0501, 0.1967, 0.0898, 0.1410, 0.1008, 0.0707, 0.0429, 0.1318,
         0.0699],
        [0.0854, 0.0705, 0.1120, 0.1468, 0.0726, 0.0905, 0.0514, 0.0475, 0.0942,
         0.2291],
        [0.1099, 0.0408, 0.1151, 0.1131, 0.0953, 0.0990, 0.1039, 0.0353, 0.1833,
         0.1044],
        [0.1086, 0.0750, 0.1088, 0.0851, 0.1063, 0.1355, 0.0979, 0.0539, 0.1477,
         0.0812],
        [0.1432, 0.0573, 0.0804, 0.0793, 0.0975, 0.1443, 0.0898, 0.0560, 0.1377,
       

In [5]:
import torch
from torch.utils.data import TensorDataset, DataLoader, Subset
from tensorflow import keras

# 1. Load data directly via Keras
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# 2. Normalize images (0-255 -> 0.0-1.0) and add channel dimension for PyTorch (N, C, H, W)
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1) / 255.0
y_train = torch.tensor(y_train, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1) / 255.0
y_test = torch.tensor(y_test, dtype=torch.long)

# 3. Create PyTorch datasets
full_train = TensorDataset(X_train, y_train)
full_test = TensorDataset(X_test, y_test)

# 4. Subset and load into DataLoaders
train_data = Subset(full_train, range(5000))
test_data = Subset(full_test, range(1000))

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

In [6]:
image, label=train_data[0]
image.max(), label

(tensor(1.), tensor(9))

In [7]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(), lr=1e-3)

In [8]:
train_loss = []
test_accuracy = []
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    seen = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)  

        optimizer.zero_grad()
        preds = model(images)
        loss = loss_fn(preds, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        seen += images.size(0)

    model.eval()
    correct = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            preds = model(images)
            correct += (preds.argmax(1) == labels).sum().item()

    train_loss.append(running_loss / seen)
    test_accuracy.append(correct / len(test_data))

    print(
        f"epoch {epoch + 1}/{epochs}, "
        f"loss: {train_loss[-1]:.4f}, "
        f"Test: {test_accuracy[-1]:.4f}"
    )

epoch 1/10, loss: 2.2973, Test: 0.1550
epoch 2/10, loss: 2.2925, Test: 0.1780
epoch 3/10, loss: 2.2869, Test: 0.2170
epoch 4/10, loss: 2.2810, Test: 0.2540
epoch 5/10, loss: 2.2746, Test: 0.2810
epoch 6/10, loss: 2.2679, Test: 0.3000
epoch 7/10, loss: 2.2607, Test: 0.3130
epoch 8/10, loss: 2.2529, Test: 0.3230
epoch 9/10, loss: 2.2450, Test: 0.3420
epoch 10/10, loss: 2.2376, Test: 0.3650


In [9]:
torch.manual_seed(0)
 
bn_model=nn.Sequential(
    nn.Flatten(),
    nn.Linear(784,128),nn.BatchNorm1d(128),nn.ReLU(),
    nn.Linear(128,128),nn.BatchNorm1d(128),nn.ReLU(),
    nn.Linear(128,128),nn.BatchNorm1d(128),nn.ReLU(),
    nn.Linear(128,64),nn.BatchNorm1d(64),nn.ReLU(),
    nn.Linear(64,64),nn.BatchNorm1d(64),nn.ReLU(),
    nn.Linear(64,64),nn.BatchNorm1d(64),nn.ReLU(),
    nn.Linear(64,10),
).to(device)
 
print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=128, bias=True)
  (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (6): ReLU()
  (7): Linear(in_features=64, out_features=10, bias=True)
  (8): Softmax(dim=1)
)


In [10]:
loss_fn=loss_fn

In [11]:
optimizer=torch.optim.SGD(bn_model.parameters(), lr=0.005, momentum=0.9)
linear_layers=[1,4,7,10,13,16]
bn_gradients={i:[] for i in linear_layers}

for epoch in range(10):
    bn_model.train()
    
    for images, labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)
        
        optimizer.zero_grad()
        preds=bn_model(images)
        loss=loss_fn(preds,labels)
        loss.backward()
        
        for i in linear_layers:
            bn_gradients[i].append(bn_model[i].weight.grad.clone().cpu().numpy())
            
        optimizer.step()
    print(f"Epoch{epoch+1}/10 completed.")
    print("last batch grad size per layer:")
    for i in linear_layers:
        print(f"Layer {i}:\n{bn_gradients[i][-1]}")

Epoch1/10 completed.
last batch grad size per layer:
Layer 1:
[[ 0.          0.          0.         ... -0.01467568  0.
   0.        ]
 [ 0.          0.          0.         ...  0.00116841  0.
   0.        ]
 [ 0.          0.          0.         ...  0.00029772  0.
   0.        ]
 ...
 [ 0.          0.          0.         ... -0.00052202  0.
   0.        ]
 [ 0.          0.          0.         ... -0.0014238   0.
   0.        ]
 [ 0.          0.          0.         ... -0.00359056  0.
   0.        ]]
Layer 4:
[[-0.01339127  0.00275425  0.00148362 ...  0.0231845  -0.0011008
   0.00851429]
 [-0.02200258  0.07875534 -0.01652026 ... -0.06636047 -0.02870031
  -0.072639  ]
 [ 0.00928762 -0.0535978   0.0117621  ... -0.00413978  0.01719081
   0.00069623]
 ...
 [-0.01451185  0.03347595 -0.00239522 ... -0.03321301  0.0046324
  -0.04612309]
 [ 0.02061052  0.01755725 -0.00430194 ... -0.01955064  0.00416614
   0.00243422]
 [ 0.04450702 -0.00764187 -0.01508671 ...  0.00529252 -0.01017149
  -0.000664

In [12]:
# VGG

vgg = nn.Sequential(
    #Block 1
    nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    #Block 2
    nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    #Block 3
    nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    #Block 4
    nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    #Block 5
    nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),
    nn.Linear(512 * 7 * 7, 4096), nn.ReLU(),
    nn.Linear(4096, 4096), nn.ReLU(),
    nn.Linear(4096, 1000)
).to(device)

In [13]:
class VGG(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.features

In [14]:
import torch.nn.functional as F
from tqdm.notebook import tqdm

In [15]:
vgg_loss_fn = nn.CrossEntropyLoss()
vgg_optimizer = torch.optim.SGD(vgg.parameters(), lr=1e-3, momentum=0.9)
vgg_epochs = 1

for epoch in range(vgg_epochs):
    vgg.train()
    running_loss = 0.0
    seen = 0

    train_progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{vgg_epochs} [train]", leave=False)
    for images, labels in train_progress:
        images = F.interpolate(images, size=(224, 224), mode="bilinear", align_corners=False)
        images = images.repeat(1, 3, 1, 1).to(device)
        labels = labels.to(device)

        vgg_optimizer.zero_grad()
        preds = vgg(images)[:, :10]
        loss = vgg_loss_fn(preds, labels)
        loss.backward()
        vgg_optimizer.step()

        running_loss += loss.item() * images.size(0)
        seen += images.size(0)
        train_progress.set_postfix(loss=f"{loss.item():.4f}")

    vgg.eval()
    correct = 0
    test_progress = tqdm(test_loader, desc=f"Epoch {epoch + 1}/{vgg_epochs} [test]", leave=False)
    with torch.no_grad():
        for images, labels in test_progress:
            images = F.interpolate(images, size=(224, 224), mode="bilinear", align_corners=False)
            images = images.repeat(1, 3, 1, 1).to(device)
            labels = labels.to(device)
            correct += (vgg(images)[:, :10].argmax(1) == labels).sum().item()

    print(
        f"epoch {epoch + 1}/{vgg_epochs}, "
        f"loss: {running_loss / seen:.4f}, "
        f"Test: {correct / len(test_data):.4f}"
    )

Epoch 1/1 [train]:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch 1/1 [test]:   0%|          | 0/8 [00:00<?, ?it/s]

epoch 1/1, loss: 2.3027, Test: 0.1110
